In [1]:
# use this block to produce root files starting from a variable number of txt files from the parameter analyzer. It works with txt files with the header set by the autosave mode on the parameter analyzer, using the header that we agreed on. This block includes IBACK, pad currents, GR current. Updated January 15th, 2026.

import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd

file = root.TFile("/Users/icosivi/Desktop/pippo.root", "RECREATE")
tree = root.TTree("Tree","Tree")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/FBK/FBK_sensor-col-row.xlsx')

event = array('i', [0])
wafer = array('i', [0])
row = array('i', [0])
column = array('i', [0])
sensor = array('i', [0])
date = str()

#V = root.std.vector("double")()
V = root.std.vector("float")()
IBACK = root.std.vector("float")()
IPAD1 = root.std.vector("float")()
IPAD2 = root.std.vector("float")()
IPAD3 = root.std.vector("float")()
IGR = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("date", date, "date/C")

V.reserve(1000)
IBACK.reserve(1000)
IPAD1.reserve(1000)
IPAD2.reserve(1000)
IPAD3.reserve(1000)
IGR.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)
tree.Branch("I_PAD1", "std::vector<float>", IPAD1)
tree.Branch("I_PAD2", "std::vector<float>", IPAD2)
tree.Branch("I_PAD3", "std::vector<float>", IPAD3)
tree.Branch("IGR", "std::vector<float>", IGR)

txt_files = []

txt_files.append(glob.glob("/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/*.txt"))
#txt_files.append(glob.glob("/Users/icosivi/Desktop/PRE-SERIE/HPK/Torino_tests/16x16/Batch2/*.txt"))

for t in txt_files:
    print(t)

start_line = 114  #Depending on the device analyzer, values may start from line 114 (PSSA) or 116 (PSF), choose the value accordingly. And do not mix files from different device analyzers!
 
counter = 0

for t in txt_files:
 for txt in t:
    V.clear()
    IBACK.clear()
    IPAD1.clear()
    IPAD2.clear()
    IPAD3.clear()
    IGR.clear()
    #CP.clear()

    event[0] = counter

    with open(txt,"r") as t:
        lines_list = t.readlines()
        lineo = lines_list[0]
        header_line = re.split(" |\t",lineo)
        header = re.split('_|-|\n',header_line[2])
        print(header)
        wafer[0] = int( re.findall(r'\d+', header[4])[0] )
        column[0] = int( re.findall(r'\d+', header[5])[0] )
        row[0] = int( re.findall(r'\d+', header[6])[0] )
        sensor[0] = int(wb[(wb['Row'] == row[0]) & (wb['Column'] == column[0])]['Sensor'].iloc[0])
        
        
        linetwo = lines_list[2]
        date_line = re.split(" |\t|\n",linetwo)
        #print(date_line)
        date = str(date_line[2])
        
        lines = lines_list[start_line:]

        for k, line in enumerate(lines):
            #print(line)
            if float(line.split("	")[0])<0:
                if( V.size()>3):
                    if( -1*float(line.split("	")[0])>-1*float(lines[k-1].split("	")[0]) ):
                    #if( -1*float(line.split("	")[0])>float(V.at( V.size()-1 )) ):
                        V.push_back( -1*float(line.split("	")[0]) )
                    else:
                        break
                else:
                    V.push_back( -1*float(line.split("	")[0]) )
            else:
                if( V.size()>3):
                    if( float(line.split("	")[0])>float(lines[k-1].split("	")[0]) ):
                    #if( float(line.split("	")[0])>float(V.at( V.size()-1 )) ):
                        V.push_back( float(line.split("	")[0]) )
                    else:
                        break
                else:
                    V.push_back( float(line.split("	")[0]) )

            if float(line.split("	")[1])<0:
                IBACK.push_back( -1*float(line.split("	")[1]) )
            else:
                IBACK.push_back( float(line.split("	")[1]) )
                
            if float(line.split("	")[2])<0:
                IPAD1.push_back( -1*float(line.split("	")[2]) )
            else:
                IPAD1.push_back( float(line.split("	")[2]) )
                
            if float(line.split("	")[3])<0:
                IPAD2.push_back( -1*float(line.split("	")[3]) )
            else:
                IPAD2.push_back( float(line.split("	")[3]) )
            
            if float(line.split("	")[4])<0:
                IPAD3.push_back( -1*float(line.split("	")[4]) )
            else:
                IPAD3.push_back( float(line.split("	")[4]) )
            
            if float(line.split("	")[6])<0:
                IGR.push_back( -1*float(line.split("	")[6]) )
            else:
                IGR.push_back( float(line.split("	")[6]) )

    tree.Fill()
    counter += 1

tree.Write()
file.Write()
file.Close()

Welcome to JupyROOT 6.30/04
['/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W16_5-2 [(6) _ 5_19_2026 12_09_09 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W3_8-5 [(4) _ 6_4_2026 11_52_13 AM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W3_4-4 [(8) _ 6_4_2026 12_18_25 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W3_6-1 [(14) _ 6_4_2026 2_08_29 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W16_4-2 [(8) _ 5_19_2026 12_24_44 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W16_3-1 [(7) _ 5_19_2026 12_17_54 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W16_8-3 [(10) _ 5_19_2026 12_40_23 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/FBK/Torino_tests/16x16/PRE_LF-FBK_16x16_W3_6-2 [(11) _ 6_4_2026 1_49_38 PM].txt', '/Users/icosivi/Desktop/PRE-SERIE/

In [ ]:
file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_16x16_IV.root", "RECREATE")
#file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/pippo.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])
temperature = array('f', [0])
medium = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("temperature", temperature, 'temperature/F')
tree.Branch("medium", medium, 'medium/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)

csv_files = glob.glob("/Users/icosivi/Desktop/PRE-SERIE/HPK/on-wafer_data/*.xlsx")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_sensor-col-row.xlsx')

nevent = 0

for csv in csv_files:
  #print(csv)
  #print(get_text_from_green_cells(csv))
  list_of_medium = get_text_from_green_cells(csv)
  df = pd.read_excel(csv, sheet_name='IV', header=None)
  nome_senza_ext = os.path.splitext(csv)[0]
  wafer[0] = int(nome_senza_ext.split('-')[-1])

  for i in range(24):
      V.clear()
      IBACK.clear()
      
      event[0] = nevent
      sensor[0] = int(df.iat[0,i+1])
      c, r = wb.loc[wb['Sensor'] == int(i+1), ['Column', 'Row']].values[0]
      column[0] = int(c)
      row[0] = int(r)
      
      if int(df.iat[0,i+1]) in list_of_medium:
        medium[0] = 1
      else:
        medium[0] = 0
      
      I_list = df.iloc[3:,i+1].dropna().tolist()
      if I_list:
        temperature[0] = float(df.iat[1,i+1])
        subset = df.iloc[3:, [0, i+1]].dropna() 
        for _, rr in subset.iterrows():  
          IBACK.push_back( float(rr.iloc[1]) )
          V.push_back( float(rr.iloc[0]) )
        
      tree.Fill()
      nevent += 1
      
      
      #V_list = df[3:,0].dropna().tolist()
      #for v in V_list:
      #  V.push_back( float(v) )

tree.Write()
file.Write()
file.Close()
      
      
      
      
      
  

In [ ]:
def to_binary(number, bit_length):
    """
    Converte un numero decimale in una stringa binaria di lunghezza fissa.
    
    Argomenti:
        number (int): Il numero decimale da convertire.
        bit_length (int): Il numero di bit desiderati in output.
        
    Ritorna:
        str: La rappresentazione binaria invertita (Little Endian) del numero.
    """
    # Converte il numero in binario standard (es: "0b1"), 
    # rimuove il prefisso '0b' e aggiunge zeri a sinistra fino alla lunghezza desiderata
    binary_standard = bin(number)[2:].zfill(bit_length)
    
    # Se il numero originale era più lungo del bit_length, tronchiamo per sicurezza
    binary_fixed = binary_standard[-bit_length:]
    
    # Invertiamo la stringa come richiesto dall'esempio (1 -> 10000 invece di 00001)
    return binary_fixed[::-1]


def to_decimal(binary_str):
    """
    Converte una stringa binaria invertita (Little Endian) nel suo valore decimale.
    Esempio: "10000" -> 1
    """
    # Inverte la stringa per riportare il bit meno significativo (LSB) alla fine
    binary_standard = binary_str[::-1]
    
    # Converte la stringa binaria standard in un intero decimale
    return int(binary_standard, 2)

In [ ]:
# producer of the xls to register components on the database for the 16x16
xl_filename="HPK_16x16_PRE-SERIES"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_16x16_IV.root")
tree_qa = file_qa.Get("Tree")

for j,event in enumerate(tree_qa):
    
    vendor_bit = 0
    wafer_bit = to_binary(event.wafer,14)
    sensor_bit = to_binary(event.sensor,5)
    wws["A%i" %(j+2)] = 'PRE'+str(vendor_bit)+str(wafer_bit)+str(sensor_bit)
    
    wws["B%i" %(j+2)] = 'HPK'
    wws["C%i" %(j+2)] = None
    wws["D%i" %(j+2)] = event.wafer
    wws["E%i" %(j+2)] = '16x16'
    
    #Column, Row = wb.loc[df['Sensor'] == sensor_number, ['Column', 'Row']].values[0]

    wws["F%i" %(j+2)] = event.row
    wws["G%i" %(j+2)] = event.column
    wws["H%i" %(j+2)] = event.sensor

save_path = '/Users/icosivi/Desktop/PRE-SERIE/HPK/'
wb.save(save_path+xl_filename+".xlsx")

In [ ]:
import xlwings as xw
import os

def update_gelpak_live(wafer, sensor, gelpak_label, observations=""):
    """
    Aggiorna la colonna 'Gelpak' e 'Observations' in tempo reale se il file è aperto in Excel.
    Verifica l'unicità della label prima di scrivere.
    """
    
    file_name = '/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_16x16_PRE-SERIES_DSF-optical-inspection.xlsx'
    #file_name = '/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_QC-TS_PRE-SERIES_DSF-optical-inspection.xlsx'
    
    try:
        # Tenta di connettersi alla cartella di lavoro se è già aperta
        try:
            wb = xw.Book(file_name)
        except Exception:
            # Se non è aperta, la apre
            if os.path.exists(file_name):
                wb = xw.Book(file_name)
            else:
                print(f"Errore: Il file '{file_name}' non esiste.")
                return

        sheet = wb.sheets[0] # Assume che sia il primo foglio

        # Legge tutti i dati correnti per processarli
        data = sheet.range('A1').expand().value
        
        # Identifica gli indici delle colonne basandosi sull'intestazione (prima riga)
        header = data[0]
        try:
            idx_wafer = header.index('Wafer')
            idx_sensor = header.index('Sensor Number')
            idx_gelpak = header.index('Gelpak')
            idx_obs = header.index('Observations')
        except ValueError as e:
            print(f"Errore: Colonne non trovate. Verifica le intestazioni: {e}")
            return

        # 1. CONTROLLO UNICITÀ GLOBALE
        existing_labels = [row[idx_gelpak] for row in data[1:] if row[idx_gelpak] is not None]
        
        if gelpak_label in existing_labels:
            print(f"ERRORE: La label '{gelpak_label}' è già presente. Operazione annullata.")
            return

        # 2. RICERCA RIGA E AGGIORNAMENTO
        found = False
        for i, row in enumerate(data[1:], start=2): # start=2 per conteggio Excel
            if row[idx_wafer] == wafer and row[idx_sensor] == sensor:
                # Scrive la label Gelpak
                sheet.range((i, idx_gelpak + 1)).value = gelpak_label
                
                # Scrive le osservazioni (se fornite o sovrascrive con stringa vuota)
                sheet.range((i, idx_obs + 1)).value = observations
                
                print(f"Riga {i} aggiornata: Gelpak={gelpak_label}, Obs={observations}")
                found = True
                break
        
        if not found:
            print(f"ERRORE: Nessuna riga trovata per Wafer {wafer} e Sensore {sensor}.")
        
    except Exception as e:
        print(f"Si è verificato un errore: {e}")

In [ ]:
update_gelpak_live(83, 23, 'HPK IRRAD 8 FULL-SIZE W83 - 8')

In [ ]:
for i in range(8):
    update_gelpak_live(83, int(i+1), 'HPK IRRAD 8 QC-TS W83 - '+str(int(i+1)))